# CVM DATA MINER
## 1. Libraries and constants

In [41]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import zipfile
import io
DFP_URL = "https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/DFP/DADOS/"
CAPEX_KEYWORDS = "imobiliz|intang"
SALE_KEYWORDS = "venda|aliena|baixa|receb"
pd.set_option("display.max_rows", None)

## 2. Functions

In [78]:
def find_zips(url):
    """Return a dictionary mapping each year to its zip URL."""
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    tags = soup.find_all("a")
    zip_dict = {}
    for tag in tags:
        href = tag.get("href")
        if not href:
            continue
        if href.endswith(".zip"):
            year = int(href.removesuffix(".zip")[-4:])
            link = urljoin(url, href)
            zip_dict[year] = link
    return zip_dict

def download_zip(zip_link):
    """Download a zip from the given URL and return it as an in-memory buffer"""
    response = requests.get(zip_link, timeout=60)
    response.raise_for_status()
    return io.BytesIO(response.content)

def open_csv(zip_file, statement, basis, year, doc_type="dfp"):
    """Read a statement CSV from a CVM zip (DFP or ITR) and return a DataFrame."""
    with zipfile.ZipFile(zip_file) as z:
        csv_name = f"{doc_type}_cia_aberta_{statement}_{basis}_{year}.csv"
        df = pd.read_csv(z.open(csv_name), sep=";", encoding="latin-1")
    return df

def split_by_company(df):
    """Clean a CVM statement and split it by company.

    Keeps the latest fiscal year, drops duplicated rows and converts
    values to thousands of reais. Returns a dictionary mapping each
    CD_CVM to its DataFrame.
    """
    df_clean = df[(df["ORDEM_EXERC"] == "ÚLTIMO") & (~df.duplicated())]
    is_unit = df_clean["ESCALA_MOEDA"] == "UNIDADE"
    df_clean.loc[is_unit, "VL_CONTA"] /= 1000
    df_clean.loc[is_unit, "ESCALA_MOEDA"] = "MIL"
    companies_data = {}
    for code, group in df_clean.groupby("CD_CVM"):
        companies_data[code] = group
    return companies_data

def find_capex(company):
    """Return the total capex of one company.

    Values are in thousands of reais (negative = cash out).
    """
    investment = company[company["CD_CONTA"].str.startswith("6.02.")]
    is_asset = investment["DS_CONTA"].str.contains(CAPEX_KEYWORDS, case=False)
    is_sale = investment["DS_CONTA"].str.contains(SALE_KEYWORDS, case=False)
    capex = investment[is_asset & ~is_sale]
    return capex["VL_CONTA"].sum()

In [79]:
year = 2025
statement = "DFC_MI"
basis = "con"
doc_type = "dfp"
zip_links = find_zips(DFP_URL)
zip_data = download_zip(zip_links[year])
df = open_csv(zip_data, statement, basis, year, doc_type)

In [80]:
companies_data = split_by_company(df)
test = companies_data[24805]
print(find_capex(test))

-80443.288


In [81]:
vivara = companies_data[24805]
vivara_investing = vivara[vivara["CD_CONTA"].str.startswith("6.02.")]
is_asset = vivara_investing["DS_CONTA"].str.contains("imobiliz|intang", case=False)
is_sale = vivara_investing["DS_CONTA"].str.contains("venda|aliena|baixa|receb", case=False)
vivara_capex = vivara_investing[is_asset & ~is_sale]
capex = vivara_capex["VL_CONTA"].sum()
print(capex)

-80443.288
